# Exercise 1: Write a mapping skill and correct its map

**Goal:** After this exercise, you can map a codebase with your own skill, and correct the map.

**Scenario:** A nightly batch job works out a late-payment penalty for each account. Someone ported it to Python, and later today you add a rule to it. You write a skill that makes Codex map the job. Then you check the map against the import graph.

**Coding agent:** yes. You use Codex in a terminal.

**Where you work:** the folder `week-02/day-02/exercise-01-write-a-mapping-skill/lab` in the course folder. It is a Git repository of its own. You run the cells of this notebook and some commands in a terminal.

**Done when:** cell 6 prints `Done:`.

**Optional part:** add this step to your skill: `Start a subagent to read the modules, and use its summary.` A subagent is OpenAI's word for a helper agent.

**What you have:** in `lab`, the job's package `payment_job/` with six modules, its made-up data in `data/`, and `.agents/skills/map-codebase/SKILL.md`, a skill with its name only.

**Start from:** step 1. To start again, run the reset command of step 1 again. It builds `lab` again and deletes your commits there.

### Step 1. See the start

Open a terminal (**Terminal > New Terminal**). It opens in the course folder. Run:

```sh
source .venv/bin/activate
python3 reset.py exercise-01-write-a-mapping-skill
```

You should see `exercise-01-write-a-mapping-skill is back as it was supplied.` Then run cell 1. You should see the job's files and one commit, `The payment job, as the course supplies it`.

In [ ]:
# 1. See the start. You do not need to read this code.
import subprocess
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if not (lab / ".git").is_dir():
    print("lab is not there yet. Run the two commands of step 1 in a terminal.")
else:
    files = subprocess.run(["git", "ls-files"], cwd=lab, capture_output=True, text=True).stdout.split()
    print("\n".join(f for f in files if f.startswith(("payment_job/", ".agents/"))))
    print(subprocess.run(["git", "log", "--oneline"], cwd=lab, capture_output=True, text=True).stdout)

### Step 2. Write the skill

In the Explorer, open `lab/.agents/skills/map-codebase/SKILL.md`. Replace each `TODO` with your own words. Your skill must say these things:

| What the skill says | Why |
|---|---|
| After `description:`, when to use the skill, and that it writes `CODEMAP.md` | Codex decides from the name and the description when to use the skill: "ChatGPT and Codex start with each skill’s name and description, then load the full SKILL.md instructions when they decide to use that skill." |
| Steps, each a command, such as `Find the entry point.`: find the entry point, read the imports of each module, write `CODEMAP.md` | "Write imperative steps with explicit inputs and outputs." "So map a package from `__main__.py`, following its imports." |
| Five sections in `CODEMAP.md`: `## Flow`, `## Owners`, `## Imports`, `## Checks`, `## Risky spots` | "A **map** says which module owns what, how a request flows, where the code checks data, and where the risky spots are." |
| Under `## Imports`, one line for each import between the modules, as `a.py -> b.py` | Cell 4 compares these lines with the import graph. It reads each `a.py -> b.py` in a line, with any text around it. |

Save the file.

<details><summary>Stuck? One way to write it</summary>

```markdown
---
name: map-codebase
description: Map a Python codebase before an edit. Use when asked to map, onboard to or explain the structure of a codebase. Writes CODEMAP.md; changes no code.
---

# Map a codebase

Input: the Python package in this folder. Output: `CODEMAP.md` in this folder. Do not change any other file.

1. Find the entry point: the package's `__main__.py`, or the file with `if __name__ == "__main__":`.
2. Read the first lines of each module for its imports.
3. Write `CODEMAP.md` with these five sections:
   - `## Flow`: how one run goes through the modules, from the entry point.
   - `## Owners`: each module, and what it owns, in one line.
   - `## Imports`: one line for each import between the package's modules, as `a.py -> b.py`.
   - `## Checks`: each place that checks data, as file and line.
   - `## Risky spots`: code that runs at import, side effects such as file writes, and code without tests.
4. End with the files to read next.
```

</details>

### Step 3. Start Codex on your skill

In the terminal, run:

```sh
cd week-02/day-02/exercise-01-write-a-mapping-skill/lab
codex
```

If Codex asks if you trust this folder, answer yes. The course gave you this folder. Type `$map-codebase Map this codebase.` Then press Enter. The `$` and the name tell Codex to use your skill. If Codex asks to run a command, answer yes.

While Codex works, read your `SKILL.md` again. Codex prints each file it reads. Match that to a step of your skill.

When Codex reports that it wrote `CODEMAP.md`, type `/quit`. Then press Enter. Then run cell 2. You should see the headings of `CODEMAP.md`.

In [ ]:
# 2. See the headings of the map. You do not need to read this code.
from pathlib import Path

codemap = Path(_dh[0]) / "lab" / "CODEMAP.md"
if not codemap.is_file():
    print("CODEMAP.md is not there yet. Do step 3.")
else:
    text = codemap.read_text(encoding="utf-8")
    print("\n".join(line for line in text.splitlines() if line.startswith("#")))
    print(len(text.splitlines()), "lines. Open lab/CODEMAP.md in the Explorer to read it.")

### Step 4. Keep Codex's map in a commit

In cell 3, change `commit = True` to `commit = True`. Then run the cell. The commit keeps the map as Codex wrote it. Your correction in step 6 then shows as a change to it. You should see a line that ends in `Map by Codex, as written`.

In [ ]:
# 3. Commit Codex's map as it is.
commit = True

import subprocess
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if commit and (lab / ".git").is_dir():
    subprocess.run(["git", "add", "CODEMAP.md", ".agents/skills/map-codebase/SKILL.md"], cwd=lab)
    done = subprocess.run(["git", "commit", "-m", "Map by Codex, as written"], cwd=lab, capture_output=True, text=True)
    print(done.stdout.splitlines()[0] if done.returncode == 0 else done.stdout + done.stderr)
else:
    print("Nothing committed. Set commit = True first.")

### Step 5. Check the map against the import graph

In cell 4, set `command` to this command. Then run the cell.

| Type | Why |
|---|---|
| `command = "pydeps payment_job --max-bacon 0"` | pydeps reads the code that Python runs: "pydeps finds imports by looking for import-opcodes in python bytecodes". It does not guess. `--max-bacon 0` keeps every module: by default pydeps will "exclude nodes that are more than n hops away (default=2, 0 -> infinite)". |

The cell adds `--show-deps --no-output`, so pydeps prints the graph as text and draws no picture. You should see three lines: `Arrows in the graph:` and their number, then the arrows in the map only, then the arrows in the graph only. Decide what to correct:

| What you find | Correct this | Why |
|---|---|---|
| An arrow in the map, not in the graph | Open the file the arrow starts from, and look for the import. If it is not there, delete the line. | "If the map says `writer.py` uses `rules.py` and the graph has no such arrow, check that claim." |
| An arrow in the graph, not in the map | Add that line to `## Imports` | The map missed an import. Look for an import inside a function, not in the first lines. |
| Under `## Checks`, a line that only works out the penalty, such as the cap | Delete each such line | Open the file and line that the map names. "Validation is the code that checks data." A line that works out the penalty is the rule, not a check. |
| `## Risky spots` does not name `settings.py` | Add it | Its top-level code reads a file. "A line there that opens a file runs the first time anything imports the module." |
| Every claim that you opened holds | Add a heading `## Checked`, with one line for each claim you opened | The second commit then holds your check of the map. |

In [ ]:
# 4. Compare the map with the import graph. You do not need to read the code below the first line.
command = "pydeps payment_job --max-bacon 0"

%run "{_dh[0]}/compare_map.py" {command}

### Step 6. Correct the map, and commit

Make each correction in `lab/CODEMAP.md` in the editor. Save the file. Run cell 4 again. Both lists should show `none`.

In cell 5, change `commit = True` to `commit = True`. Then run the cell. You should see a line that ends in `Correct the map`.

In [ ]:
# 5. Commit your correction.
commit = True

import subprocess
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if commit and (lab / ".git").is_dir():
    subprocess.run(["git", "add", "CODEMAP.md"], cwd=lab)
    done = subprocess.run(["git", "commit", "-m", "Correct the map"], cwd=lab, capture_output=True, text=True)
    print(done.stdout.splitlines()[0] if done.returncode == 0 else done.stdout + done.stderr)
else:
    print("Nothing committed. Set commit = True first.")

### Step 7. Check

Run cell 6. It prints `Done:` when these things are true:

- `SKILL.md` has a description, and steps that name `CODEMAP.md`.
- Two commits hold `CODEMAP.md`: Codex's map, then your correction.
- The map's `## Imports` match the graph, and its `## Risky spots` name `settings.py`.

Otherwise it prints `Not yet:` and each thing still to do. You do not need to read its code.

In [ ]:
# 6. Check. You do not need to read this code.
%run "{_dh[0]}/check.py"

### Afterwards, together

1. What did the map get wrong, and how did you find out?
2. "The graph shows only who imports whom, not the request flow or where the code checks data." Would the import graph catch each thing you corrected? Why, or why not?
3. A study of code agents found that one model "recalled 69% of IMPORTS edges". When would you trust a map without the graph?

<details><summary>Answer</summary>

The skill: Codex reads only the name and the description until it uses the skill. So the description says when to use the skill and what it writes. The steps are commands with an input and an output, from the entry point through the imports.

Keep Codex's map first: the second commit then shows your correction as a change.

The graph: pydeps reads the bytecode, so it finds the import inside `write_report` in `writer.py`, which the first lines of the module do not show.

The kept run's map had every import right. Its wrong claims were two lines under `## Checks`: `rules.py:23` returns 0 when the account owes nothing, and `rules.py:25` caps the months. Both work out the penalty, which is the rule, not a check of data. The correction deleted them.

The skill and both maps are in [`answers/exercise-01-write-a-mapping-skill`](../answers/exercise-01-write-a-mapping-skill/). Each reason, and where each quoted line comes from, is in [`decisions.md`](../answers/exercise-01-write-a-mapping-skill/decisions.md).

</details>